<a href="https://colab.research.google.com/github/visal1411/INTERNSHIP/blob/ML/IsolationForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Cattle Health Anomaly Detection - Isolation Forest Training Script
====================================================================
Trains an unsupervised IsolationForest on cattle weight data to flag
statistical outliers (potential sickness, pregnancy, or measurement
errors) for human/vet review.

Input : cattle_weight_dataset.csv  (columns: ID, Breed, Age, Gender, Weight, ...)
Output: cattle_weight_with_anomalies.csv (adds normalized features,
        anomaly scores, flags, and a per-row SHAP explanation)

Run:  python3 train_isolation_forest.py
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

import shap

# ---------------------------------------------------------------------------
# 1. Config
# ---------------------------------------------------------------------------
INPUT_CSV = "cattle_weight_dataset.csv"
OUTPUT_CSV = "cattle_weight_with_anomalies.csv"

CONTAMINATION = 0.06     # assumed fraction of herd that's anomalous -> tune
                          # this using real vet-confirmed sickness/pregnancy
                          # rates once available. Too low = misses cases,
                          # too high = floods the farmer with false alarms.
N_ESTIMATORS = 300
RANDOM_STATE = 42

In [2]:
# ---------------------------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = {"Breed", "Age", "Gender", "Weight"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Input CSV is missing required columns: {missing}")

In [3]:
# ---------------------------------------------------------------------------
# 3. Per-breed / per-age normalization
# ---------------------------------------------------------------------------
# WHY: raw Weight alone isn't comparable across breeds or ages - a 250kg
# Local Zebu calf and a 250kg Holstein Cross adult mean very different
# things. Without normalization, IsolationForest mostly just learns
# "breed X / older cattle are heavier" rather than catching real
# individual-level anomalies.
#
# We normalize by grouping cattle into breed + age-bucket cohorts and
# expressing each animal's weight as a z-score *within its own cohort*.
# This way "unusual" means "unusual for an animal like this", not
# "unusual compared to the whole herd".

# Bucket age into ranges (in months) so each cohort has enough animals
# to compute a stable mean/std. Adjust bin edges to fit your herd's
# actual age distribution / dataset size.
AGE_BINS = [0, 12, 24, 36, 48, 60, np.inf]
AGE_LABELS = ["0-12", "13-24", "25-36", "37-48", "49-60", "60+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=AGE_BINS, labels=AGE_LABELS, right=True)

# cohort = Breed + Gender + AgeGroup
cohort_stats = (
    df.groupby(["Breed", "Gender", "AgeGroup"], observed=True)["Weight"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "CohortMeanWeight", "std": "CohortStdWeight"})
    .reset_index()
)
# guard against cohorts with only 1 animal (std = NaN or 0) -> fall back
# to a small epsilon so we don't divide by zero
cohort_stats["CohortStdWeight"] = cohort_stats["CohortStdWeight"].fillna(0).replace(0, 1e-6)

df = df.merge(cohort_stats, on=["Breed", "Gender", "AgeGroup"], how="left")

# Weight_Zscore: how many standard deviations this animal's weight is
# from the mean of its own breed/gender/age cohort. This is the
# normalized feature IsolationForest will actually train on, alongside
# Age itself (to still capture within-cohort growth trends).
df["Weight_Zscore"] = (df["Weight"] - df["CohortMeanWeight"]) / df["CohortStdWeight"]

In [4]:
# ---------------------------------------------------------------------------
# 4. Build feature matrix
# ---------------------------------------------------------------------------
# Encode categoricals. Breed/Gender are kept (in addition to the
# normalized z-score) so the model can still learn breed-specific
# variance patterns, not just the mean-centered value.
le_breed = LabelEncoder()
le_gender = LabelEncoder()

features = pd.DataFrame({
    "Breed_enc": le_breed.fit_transform(df["Breed"]),
    "Gender_enc": le_gender.fit_transform(df["Gender"]),
    "Age": df["Age"],
    "Weight_Zscore": df["Weight_Zscore"],
})

In [5]:
# ---------------------------------------------------------------------------
# 5. Train IsolationForest
# ---------------------------------------------------------------------------
iso_forest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
)
iso_forest.fit(features)

# decision_function: higher = more normal, lower/negative = more anomalous
df["AnomalyScore"] = iso_forest.decision_function(features).round(4)
raw_pred = iso_forest.predict(features)  # -1 = anomaly, 1 = normal


# ---------------------------------------------------------------------------
# 6. Direction-aware flag labels
# ---------------------------------------------------------------------------
# The model only knows "outlier" vs "not outlier" - it has NO concept of
# *why*. We add a thin rule layer on top using information the model
# itself doesn't use (direction of deviation + gender) purely to make
# the output more actionable. This is still a heuristic, not a
# diagnosis - always route flagged animals to a human/vet for the
# actual call.
def refine_flag(is_anomaly, weight_zscore, gender):
    if not is_anomaly:
        return "Normal"
    if weight_zscore < 0:
        return "Flagged - Potential Sickness"
    else:
        return "Flagged - Possible Pregnancy or Overweight" if gender == "Female" \
            else "Flagged - Unusual, Needs Review"

df["IsAnomaly"] = raw_pred == -1
df["Flag"] = [
    refine_flag(is_anom, z, g)
    for is_anom, z, g in zip(df["IsAnomaly"], df["Weight_Zscore"], df["Gender"])
]

In [6]:
# ---------------------------------------------------------------------------
# 7. SHAP explanation - which feature drove each flag
# ---------------------------------------------------------------------------
# SHAP tells you, per animal, how much each feature pushed the anomaly
# score up or down. This turns "this cow is weird" into "this cow is
# weird mainly because of its weight relative to its cohort" - which is
# what a farmer/vet can actually act on.
explainer = shap.TreeExplainer(iso_forest)
shap_values = explainer.shap_values(features)

feature_names = features.columns.tolist()
shap_df = pd.DataFrame(shap_values, columns=[f"SHAP_{c}" for c in feature_names])

# Identify the single biggest driver per row (most negative SHAP value =
# pushed the animal furthest toward "anomalous")
def top_driver(row):
    vals = row[[f"SHAP_{c}" for c in feature_names]]
    return vals.idxmin().replace("SHAP_", "")

df = pd.concat([df.reset_index(drop=True), shap_df.reset_index(drop=True)], axis=1)
df["TopAnomalyDriver"] = df.apply(top_driver, axis=1)

In [7]:
# ---------------------------------------------------------------------------
# 8. Save + summary
# ---------------------------------------------------------------------------
output_cols = [
    "ID", "Breed", "Age", "AgeGroup", "Gender", "Weight",
    "CohortMeanWeight", "Weight_Zscore",
    "AnomalyScore", "Flag", "TopAnomalyDriver",
] + [f"SHAP_{c}" for c in feature_names]

df[output_cols].to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}  ({len(df)} rows)\n")
print("Flag counts:")
print(df["Flag"].value_counts())
print("\nTop anomaly driver, among flagged animals only:")
print(df.loc[df["IsAnomaly"], "TopAnomalyDriver"].value_counts())
print("\nExample flagged rows:")
print(
    df.loc[df["IsAnomaly"], ["ID", "Breed", "Age", "Gender", "Weight",
                              "AnomalyScore", "Flag", "TopAnomalyDriver"]]
    .head(10)
    .to_string(index=False)
)

Saved: cattle_weight_with_anomalies.csv  (1600 rows)

Flag counts:
Flag
Normal                                        1504
Flagged - Potential Sickness                    41
Flagged - Unusual, Needs Review                 36
Flagged - Possible Pregnancy or Overweight      19
Name: count, dtype: int64

Top anomaly driver, among flagged animals only:
TopAnomalyDriver
Weight_Zscore    65
Age              14
Breed_enc        14
Gender_enc        3
Name: count, dtype: int64

Example flagged rows:
    ID          Breed  Age Gender  Weight  AnomalyScore                                       Flag TopAnomalyDriver
C00041     Local Zebu   34   Male   327.6       -0.0069            Flagged - Unusual, Needs Review    Weight_Zscore
C00062  Brahman Cross   72   Male   584.0       -0.0160            Flagged - Unusual, Needs Review              Age
C00074   Sindhi Cross   12   Male   140.3       -0.0318            Flagged - Unusual, Needs Review        Breed_enc
C00075     Local Zebu    6   Male    25